# Goal
Analyze the export performance of semiconductor products (specifically integrated circuits) by destination country to identify high-value markets and trends using trade data.

# Data Loading

| Column                                                 | Description                                                                                         |
| ------------------------------------------------------ | --------------------------------------------------------------------------------------------------- |
| **typeCode**                                           | Type of dataset (e.g., trade data category, usually 'C' for commodity)                              |
| **freqCode**                                           | Frequency of the data (usually annual or monthly)                                                   |
| **refPeriodId**, **refYear**, **refMonth**, **period** | Time indicators — useful for time-series analysis                                                   |
| **reporterCode**, **reporterISO**, **reporterDesc**    | Country that is reporting the trade (e.g., Taiwan)                                                  |
| **flowCode**, **flowDesc**                             | Type of trade: Import or Export                                                                     |
| **partnerCode**, **partnerISO**, **partnerDesc**       | Trade partner (who you're buying from or selling to)                                                |
| **cmdCode**                                            | HS (Harmonized System) commodity code — identifies the product (e.g., 8542 for ICs)                 |
| **cmdDesc**                                            | Product description (e.g., “Electronic integrated circuits”)                                        |
| **qty**, **netWgt**, **grossWgt**                      | Quantity and weight of traded goods (may be True/False if not filled correctly)                     |
| **fobvalue**                                           | **Free On Board** value — cost of goods at point of export (good for export-focused value)          |
| **cifvalue**                                           | **Cost, Insurance, and Freight** — includes shipping/import costs (useful for import-side analysis) |
| **motCode**, **motDesc**                               | Mode of transport (e.g., sea, air)                                                                  |
| **customsCode**, **customsDesc**                       | Customs declaration info                                                                            |
| **qtyUnitCode**, **qtyUnitAbbr**                       | Units of measurement (e.g., kg, pieces)                                                             |
| **isReported**                                         | Whether the data is officially reported (vs. estimated)                                             |

In [56]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

In [57]:
df = pd.read_csv('C:/Users/Admin/Documents/dibimbing/Dibimbing---DSDA/Final Project/TradeData_6_22_2025_11_47_42.csv', index_col=False, encoding='latin1')
df

,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,...,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,A,20200101,2020,52,2020,8,ALB,Albania,M,...,0.00,False,0.0,False,1961524.223,0.0,1961524.223,0,False,True
1,C,A,20200101,2020,52,2020,8,ALB,Albania,M,...,0.00,False,0.0,False,183993.896,0.0,183993.896,0,False,True
2,C,A,20200101,2020,52,2020,8,ALB,Albania,M,...,0.00,False,0.0,False,21676.351,0.0,21676.351,0,False,True
3,C,A,20200101,2020,52,2020,8,ALB,Albania,M,...,0.00,False,0.0,False,1647454.576,0.0,1647454.576,0,False,True
4,C,A,20200101,2020,52,2020,8,ALB,Albania,M,...,14.46,False,0.0,False,4554.309,0.0,4554.309,0,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66854,C,A,20240101,2024,52,2024,860,UZB,Uzbekistan,X,...,NaN,False,0.0,False,NaN,1852.0,1852.000,0,False,True
66855,C,A,20240101,2024,52,2024,860,UZB,Uzbekistan,X,...,NaN,False,0.0,False,NaN,46601.0,46601.000,2,False,True
66856,C,A,20240101,2024,52,2024,860,UZB,Uzbekistan,X,...,NaN,False,0.0,False,NaN,4279.0,4279.000,0,False,True
66857,C,A,20240101,2024,52,2024,860,UZB,Uzbekistan,X,...,NaN,False,0.0,False,NaN,225.0,225.000,0,False,True


# Data Understanding

In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66859 entries, 0 to 66858
Data columns (total 47 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   typeCode                  66859 non-null  object 
 1   freqCode                  66859 non-null  object 
 2   refPeriodId               66859 non-null  int64  
 3   refYear                   66859 non-null  int64  
 4   refMonth                  66859 non-null  int64  
 5   period                    66859 non-null  int64  
 6   reporterCode              66859 non-null  int64  
 7   reporterISO               66859 non-null  object 
 8   reporterDesc              66859 non-null  object 
 9   flowCode                  66859 non-null  object 
 10  flowDesc                  66859 non-null  object 
 11  partnerCode               66859 non-null  int64  
 12  partnerISO                66859 non-null  object 
 13  partnerDesc               66859 non-null  object 
 14  partne

We have 47 tables, but not all of them is useful

In [59]:
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

typeCode: 1 unique values
freqCode: 1 unique values
refPeriodId: 5 unique values
refYear: 5 unique values
refMonth: 1 unique values
period: 5 unique values
reporterCode: 169 unique values
reporterISO: 169 unique values
reporterDesc: 169 unique values
flowCode: 2 unique values
flowDesc: 2 unique values
partnerCode: 245 unique values
partnerISO: 244 unique values
partnerDesc: 245 unique values
partner2Code: 1 unique values
partner2ISO: 1 unique values
partner2Desc: 1 unique values
classificationCode: 4 unique values
classificationSearchCode: 1 unique values
isOriginalClassification: 1 unique values
cmdCode: 1 unique values
cmdDesc: 1 unique values
aggrLevel: 1 unique values
isLeaf: 1 unique values
customsCode: 1 unique values
customsDesc: 1 unique values
mosCode: 1 unique values
motCode: 1 unique values
motDesc: 1 unique values
qtyUnitCode: 3 unique values
qtyUnitAbbr: 2 unique values
qty: 8496 unique values
isQtyEstimated: 2 unique values
altQtyUnitCode: 4 unique values
altQtyUnitAbbr: 

In [60]:
for col in df.columns:
    print(f"Column: {col}")
    print(df[col].unique())
    print(df[col].nunique())
    print("-" * 40)

Column: typeCode
['C']
1
----------------------------------------
Column: freqCode
['A']
1
----------------------------------------
Column: refPeriodId
[20200101 20210101 20220101 20230101 20240101]
5
----------------------------------------
Column: refYear
[2020 2021 2022 2023 2024]
5
----------------------------------------
Column: refMonth
[52]
1
----------------------------------------
Column: period
[2020 2021 2022 2023 2024]
5
----------------------------------------
Column: reporterCode
[  8  20  24  28  31  32  36  40  44  48  51  52  56  60  68  70  72  76
  84  96 100 104 108 112 116 120 124 132 136 140 144 152 156 170 174 178
 180 188 191 192 196 203 204 208 212 214 218 222 231 233 242 246 251 258
 266 268 270 275 276 296 300 308 320 328 340 344 348 352 360 364 372 376
 380 384 388 392 398 400 404 410 414 417 418 422 426 428 430 440 442 446
 450 454 458 462 470 478 480 484 490 496 498 499 500 504 508 512 516 524
 528 531 533 554 558 562 566 579 586 591 598 600 604 608 616 62

(reportercode, reporterdesc, partnercode, partnerdesc, refYear, refMonth, flowcode, flowdesc, cmdCode, cmdDesc, fobvalue, cifvalue, netWgt, grossWgt) Apparently these are the tables that is necessary for this analysis. First, need to copy the original data.

In [61]:
df_copy = df.copy()

In [62]:
# Keep only the necessary columns for analysis
columns_to_keep = [
    'reporterCode', 'reporterDesc', 'partnerCode', 'partnerDesc',
    'refYear', 'flowCode', 'flowDesc',
    'cmdCode', 'cmdDesc', 'qty', 'fobvalue', 'cifvalue', 'netWgt', 'grossWgt', 'primaryValue'
]
df_copy = df_copy[columns_to_keep]
df_copy.head()

,reporterCode,reporterDesc,partnerCode,partnerDesc,refYear,flowCode,flowDesc,cmdCode,cmdDesc,qty,fobvalue,cifvalue,netWgt,grossWgt,primaryValue
0,8,Albania,0,World,2020,M,Import,8542,Electronic integrated circuits,0.0,0.0,1961524.223,0.00,0.0,1961524.223
1,8,Albania,156,China,2020,M,Import,8542,Electronic integrated circuits,0.0,0.0,183993.896,0.00,0.0,183993.896
2,8,Albania,276,Germany,2020,M,Import,8542,Electronic integrated circuits,0.0,0.0,21676.351,0.00,0.0,21676.351
3,8,Albania,380,Italy,2020,M,Import,8542,Electronic integrated circuits,0.0,0.0,1647454.576,0.00,0.0,1647454.576
4,8,Albania,410,Rep. of Korea,2020,M,Import,8542,Electronic integrated circuits,18.0,0.0,4554.309,14.46,0.0,4554.309


In [63]:
# Split export data
df_copy_export = df_copy[df_copy['flowDesc'] == 'Export']

# Split import data
df_copy_import = df_copy[df_copy['flowDesc'] == 'Import']


In [64]:
df_copy_export.head()

,reporterCode,reporterDesc,partnerCode,partnerDesc,refYear,flowCode,flowDesc,cmdCode,cmdDesc,qty,fobvalue,cifvalue,netWgt,grossWgt,primaryValue
943,20,Andorra,0,World,2020,X,Export,8542,Electronic integrated circuits,0.0,3238989.065,NaN,1715.7,0.0,3238989.065
944,20,Andorra,251,France,2020,X,Export,8542,Electronic integrated circuits,0.0,330554.722,NaN,164.3,0.0,330554.722
945,20,Andorra,410,Rep. of Korea,2020,X,Export,8542,Electronic integrated circuits,0.0,23.151,NaN,0.5,0.0,23.151
946,20,Andorra,568,"Other Europe, nes",2020,X,Export,8542,Electronic integrated circuits,0.0,68309.326,NaN,43.0,0.0,68309.326
947,20,Andorra,579,Norway,2020,X,Export,8542,Electronic integrated circuits,0.0,91901.028,NaN,58.0,0.0,91901.028


In [65]:
df_copy_import.head()

,reporterCode,reporterDesc,partnerCode,partnerDesc,refYear,flowCode,flowDesc,cmdCode,cmdDesc,qty,fobvalue,cifvalue,netWgt,grossWgt,primaryValue
0,8,Albania,0,World,2020,M,Import,8542,Electronic integrated circuits,0.0,0.0,1961524.223,0.00,0.0,1961524.223
1,8,Albania,156,China,2020,M,Import,8542,Electronic integrated circuits,0.0,0.0,183993.896,0.00,0.0,183993.896
2,8,Albania,276,Germany,2020,M,Import,8542,Electronic integrated circuits,0.0,0.0,21676.351,0.00,0.0,21676.351
3,8,Albania,380,Italy,2020,M,Import,8542,Electronic integrated circuits,0.0,0.0,1647454.576,0.00,0.0,1647454.576
4,8,Albania,410,Rep. of Korea,2020,M,Import,8542,Electronic integrated circuits,18.0,0.0,4554.309,14.46,0.0,4554.309


# Data Cleaning (Handling Missing Values)

In [66]:
df_copy_export.describe()

,reporterCode,partnerCode,refYear,cmdCode,qty,fobvalue,cifvalue,netWgt,grossWgt,primaryValue
count,30469.000000,30469.000000,30469.000000,30469.0,3.046900e+04,3.046900e+04,8.105000e+03,2.702300e+04,3.046900e+04,3.046900e+04
mean,459.487840,433.695395,2021.927369,8542.0,2.992673e+05,3.001732e+08,6.691367e+04,1.419498e+05,1.045598e+04,3.001732e+08
std,248.405229,256.203464,1.371447,0.0,1.121424e+07,4.999778e+09,3.059687e+06,2.194341e+06,5.637530e+05,4.999778e+09
min,8.000000,0.000000,2020.000000,8542.0,0.000000e+00,2.000000e-02,0.000000e+00,0.000000e+00,0.000000e+00,2.000000e-02
25%,246.000000,212.000000,2021.000000,8542.0,0.000000e+00,4.339100e+03,0.000000e+00,6.000000e+00,0.000000e+00,4.339100e+03
50%,442.000000,428.000000,2022.000000,8542.0,0.000000e+00,5.188784e+04,0.000000e+00,1.010180e+02,0.000000e+00,5.188784e+04
75%,703.000000,674.000000,2023.000000,8542.0,0.000000e+00,1.374973e+06,0.000000e+00,2.388971e+03,0.000000e+00,1.374973e+06
max,894.000000,899.000000,2024.000000,8542.0,1.241091e+09,2.197973e+11,1.703608e+08,1.334324e+08,8.502139e+07,2.197973e+11


In [86]:
df_copy_export.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30469 entries, 943 to 66858
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   reporterCode  30469 non-null  int64  
 1   reporterDesc  30469 non-null  object 
 2   partnerCode   30469 non-null  int64  
 3   partnerDesc   30469 non-null  object 
 4   refYear       30469 non-null  int64  
 5   flowCode      30469 non-null  object 
 6   flowDesc      30469 non-null  object 
 7   cmdCode       30469 non-null  int64  
 8   cmdDesc       30469 non-null  object 
 9   qty           30469 non-null  float64
 10  fobvalue      30469 non-null  float64
 11  cifvalue      8105 non-null   float64
 12  netWgt        27023 non-null  float64
 13  grossWgt      30469 non-null  float64
 14  primaryValue  30469 non-null  float64
dtypes: float64(6), int64(4), object(5)
memory usage: 3.7+ MB


In [67]:
df_copy_export.isnull().sum()

reporterCode        0
reporterDesc        0
partnerCode         0
partnerDesc         0
refYear             0
flowCode            0
flowDesc            0
cmdCode             0
cmdDesc             0
qty                 0
fobvalue            0
cifvalue        22364
netWgt           3446
grossWgt            0
primaryValue        0
dtype: int64

In [73]:
df_copy_export[df_copy_export.isna().any(axis=1)]

,reporterCode,reporterDesc,partnerCode,partnerDesc,refYear,flowCode,flowDesc,cmdCode,cmdDesc,qty,fobvalue,cifvalue,netWgt,grossWgt,primaryValue
943,20,Andorra,0,World,2020,X,Export,8542,Electronic integrated circuits,0.0,3238989.065,NaN,1715.7,0.0,3238989.065
944,20,Andorra,251,France,2020,X,Export,8542,Electronic integrated circuits,0.0,330554.722,NaN,164.3,0.0,330554.722
945,20,Andorra,410,Rep. of Korea,2020,X,Export,8542,Electronic integrated circuits,0.0,23.151,NaN,0.5,0.0,23.151
946,20,Andorra,568,"Other Europe, nes",2020,X,Export,8542,Electronic integrated circuits,0.0,68309.326,NaN,43.0,0.0,68309.326
947,20,Andorra,579,Norway,2020,X,Export,8542,Electronic integrated circuits,0.0,91901.028,NaN,58.0,0.0,91901.028
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66854,860,Uzbekistan,616,Poland,2024,X,Export,8542,Electronic integrated circuits,3075.0,1852.000,NaN,NaN,0.0,1852.000
66855,860,Uzbekistan,724,Spain,2024,X,Export,8542,Electronic integrated circuits,0.0,46601.000,NaN,NaN,0.0,46601.000
66856,860,Uzbekistan,762,Tajikistan,2024,X,Export,8542,Electronic integrated circuits,151.0,4279.000,NaN,NaN,0.0,4279.000
66857,860,Uzbekistan,804,Ukraine,2024,X,Export,8542,Electronic integrated circuits,6.0,225.000,NaN,NaN,0.0,225.000


In [85]:
check = df_copy_export[df_copy_export['netWgt'].isna()]
check.sample(10)

,reporterCode,reporterDesc,partnerCode,partnerDesc,refYear,flowCode,flowDesc,cmdCode,cmdDesc,qty,fobvalue,cifvalue,netWgt,grossWgt,primaryValue
50101,458,Malaysia,705,Slovenia,2023,X,Export,8542,Electronic integrated circuits,272259.0,147015.860,NaN,NaN,0.00,147015.860
50076,458,Malaysia,524,Nepal,2023,X,Export,8542,Electronic integrated circuits,0.0,33487.087,NaN,NaN,0.00,33487.087
62340,458,Malaysia,352,Iceland,2024,X,Export,8542,Electronic integrated circuits,15.0,32.121,NaN,NaN,0.00,32.121
31052,170,Colombia,0,World,2022,X,Export,8542,Electronic integrated circuits,0.0,989794.740,NaN,NaN,0.00,989794.740
23567,554,New Zealand,807,North Macedonia,2021,X,Export,8542,Electronic integrated circuits,410.0,267996.887,NaN,NaN,418.00,267996.887
61670,376,Israel,178,Congo,2024,X,Export,8542,Electronic integrated circuits,0.0,3000.000,NaN,NaN,0.00,3000.000
4892,344,"China, Hong Kong SAR",834,United Rep. of Tanzania,2020,X,Export,8542,Electronic integrated circuits,1000.0,208.322,NaN,NaN,0.00,208.322
66795,842,USA,591,Panama,2024,X,Export,8542,Electronic integrated circuits,0.0,1703276.000,NaN,NaN,0.00,1703276.000
66826,842,USA,788,Tunisia,2024,X,Export,8542,Electronic integrated circuits,183357.0,374020.000,NaN,NaN,0.00,374020.000
38047,608,Philippines,462,Maldives,2022,X,Export,8542,Electronic integrated circuits,0.0,735.000,NaN,NaN,0.57,735.000


In [82]:
mean_netWgt = df_copy_export['netWgt'].mean()
mean_netWgt

141949.75956744255

In [70]:
# Calculate and display the percentage of missing values for each column in df_copy
percent_missing = df_copy_export.isna().mean() * 100
print(percent_missing)

reporterCode     0.000000
reporterDesc     0.000000
partnerCode      0.000000
partnerDesc      0.000000
refYear          0.000000
flowCode         0.000000
flowDesc         0.000000
cmdCode          0.000000
cmdDesc          0.000000
qty              0.000000
fobvalue         0.000000
cifvalue        73.399193
netWgt          11.309856
grossWgt         0.000000
primaryValue     0.000000
dtype: float64


we don't need 'cifvalue' in here. ('fobvalue' is a cost of goods at point of export, while 'cifvalue' is a shipping/import costs). Also dropped the missing 'netWgt'

In [ ]:
# Check for zero values in selected columns of df_copy_export
zero_counts = df_copy_export.apply(lambda x: (x == 0).sum())
print(zero_counts)

reporterCode        0
reporterDesc        0
partnerCode       661
partnerDesc         0
refYear             0
flowCode            0
flowDesc            0
cmdCode             0
cmdDesc             0
qty             23336
fobvalue            0
cifvalue         8006
netWgt           1627
grossWgt        29395
primaryValue        0
dtype: int64


In [92]:
# Fill missing values in 'netWgt' with the median of the column
median_netWgt = df_copy_export['netWgt'].median()
df_copy_export['netWgt'] = df_copy_export['netWgt'].replace(0, median_netWgt)

C:\Users\Admin\AppData\Local\Temp\ipykernel_28732\2572152820.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_copy_export['netWgt'] = df_copy_export['netWgt'].replace(0, median_netWgt)


In [93]:
df_export = df_copy_export.drop(columns=['cifvalue'])

In [ ]:
# check again
percent_missing_check = df_export.isna().mean() * 100
percent_missing_check

reporterCode    0.0
reporterDesc    0.0
partnerCode     0.0
partnerDesc     0.0
refYear         0.0
flowCode        0.0
flowDesc        0.0
cmdCode         0.0
cmdDesc         0.0
qty             0.0
fobvalue        0.0
netWgt          0.0
grossWgt        0.0
primaryValue    0.0
dtype: float64

In [95]:
#check again
zero_counts = df_export.apply(lambda x: (x == 0).sum())
print(zero_counts)

reporterCode        0
reporterDesc        0
partnerCode       661
partnerDesc         0
refYear             0
flowCode            0
flowDesc            0
cmdCode             0
cmdDesc             0
qty             23336
fobvalue            0
netWgt              0
grossWgt        29395
primaryValue        0
dtype: int64


# Outlier Check

In [ ]:
# Check outliers
numerical_columns = df_copy_clean.select_dtypes(include=['int64', 'float64']).columns

plt.figure(figsize=(10, 6))
for i in range(0, len(numerical_columns)):
    plt.subplot(1,len(numerical_columns), i+1)
    sns.boxplot(y=df_copy_clean[numerical_columns[i]], color='blue')
    plt.tight_layout()

Since the operations value is the part that has outliers, we will keep the outliers, to keep the original information.

# Data Manipulation

In [ ]:
export = df_copy_clean[df_copy_clean["flowCode"] == "X"]
export.sample(5)

In [ ]:
imports = df_copy_clean[df_copy_clean["flowCode"] == "M"]
imports.sample(5)

In [ ]:
# Select only columns with float or int data types from df_copy_clean
numeric_df = df_copy_clean.select_dtypes(include=['float64', 'int64'])
numeric_df.head()

In [ ]:
def check_plot(df, variable):
    # fungsi mengambil kerangka data (df) dan
    # variabel yang diminati sebagai argumen

    # tentukan ukuran gambar
    plt.figure(figsize=(16, 4))

    # histogram
    plt.subplot(1, 3, 1)
    sns.histplot(df[variable], bins=30)
    plt.title('Histogram')

    # plot Q-Q
    plt.subplot(1, 3, 2)
    stats.probplot(df[variable], dist="norm", plot=plt)
    plt.ylabel('Variable quantiles')

    # box plot
    plt.subplot(1, 3, 3)
    sns.boxplot(y=df[variable])
    plt.title('Boxplot')

    plt.show()

In [ ]:
for col in numeric_df:
    check_plot(numeric_df, col)

There is nothing to conclude yet here.

In [ ]:
top_countries = df_copy_clean.groupby('partnerDesc')['fobvalue'].sum().sort_values(ascending=False).reset_index()
top_countries.head(10)

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=top_countries.head(10), x='fobvalue', y='partnerDesc')
plt.title('Top 10 Export Destinations for Semiconductors')
plt.xlabel('FOB Export Value')
plt.ylabel('Country')
plt.tight_layout()
plt.show()

In [ ]:
df_copy_clean.groupby(['refYear'])['fobvalue'].sum().plot(kind='bar')